# ✍️ Lab: Generative Output Evaluation Suite 🔬

## Module 05 — Evaluating Generative Outputs

---

## Welcome!

In this lab, you'll play the role of an ML engineer at a financial services firm who needs to **catch dangerous errors in LLM-generated text** before they reach decision-makers. You'll build a multi-dimensional evaluation pipeline — from extracting individual claims to running an automated LLM judge.

## 🏗️ What You'll Build

By the end of this lab you will have:
- 📊 Loaded and explored financial summaries alongside their source data
- 🔍 Extracted and verified factual claims against source documents
- ✅ Designed completeness criteria for financial domain outputs
- 🔢 Implemented numeric reasoning checks for calculations
- ⚖️ Built an LLM-as-judge evaluator with a structured rubric you design
- 🤔 Investigated cases where automated judge scores diverge from human judgment
- 📋 Combined all evaluation dimensions into an actionable quality report

## 🗺️ The Evaluation Pipeline

Here's the full pipeline you'll build, step by step:

```
┌─────────────────┐     ┌──────────────────┐     ┌────────────────────┐
│  Generated Text │────▶│ Claim Extraction  │────▶│  Factuality Check  │
│  (LLM Summary)  │     │ (Break into parts)│     │ (True? False? Made │
└─────────────────┘     └──────────────────┘     │  up? Unsupported?) │
                                                  └────────┬───────────┘
                                                           │
                        ┌──────────────────┐               ▼
                        │  LLM-as-Judge    │     ┌────────────────────┐
                        │  (Rubric-based   │◀────│   Completeness /   │
                        │   overall score) │     │   Conciseness      │
                        └────────┬─────────┘     └────────┬───────────┘
                                 │                        │
                                 ▼                        ▼
                        ┌──────────────────┐     ┌────────────────────┐
                        │  Combined Report │◀────│  Numeric Reasoning │
                        │  PASS / REVIEW / │     │  Checks            │
                        │  FAIL            │     └────────────────────┘
                        └──────────────────┘
```

Each step catches a **different kind of error** — that's why we need all of them.

## 🎯 Learning Objectives

1. Implement factuality and hallucination scoring for generated text
2. Evaluate completeness vs conciseness tradeoffs
3. Design numeric reasoning checks for calculation-heavy outputs
4. Build LLM-as-judge evaluators with structured rubrics
5. Interpret generative evaluation results for production decisions

## Prerequisites

- Module 01 (Modern AI Evaluation Foundations) completed
- Module 02 (Failure Mode Analysis) completed
- Understanding of LLM behavior and common failure modes
- Familiarity with prompt engineering
- Python proficiency (pandas, numpy)

## ⏱️ Estimated Time

**25–30 minutes** hands-on

## 📖 Scenario

You are an ML engineer at a financial services firm. Your team deployed an LLM to generate quarterly earnings analysis summaries from raw financial data. These summaries are sent to portfolio managers who use them to make investment decisions.

Recently, a portfolio manager noticed a summary that reported "revenue grew 15%" when the actual data showed 8% growth. Another summary omitted a critical disclosure about debt covenant violations. These aren't just quality issues — they're **compliance and financial risk** issues.

> ⚠️ **Why This Matters:** In finance, a single wrong number or missing disclosure can lead to million-dollar losses or regulatory penalties. Evaluation isn't optional — it's a safety net.

Your job: build an evaluation pipeline that catches factual errors, missing information, numeric mistakes, and overall quality problems **before** summaries reach decision-makers.

---

## Task 1: Environment Setup and Dependencies 🛠️

### Goal
Install and import the required packages for text analysis, evaluation scoring, and data manipulation.

### Why Start Here?
Before we can analyze any generated text, we need our toolkit ready. We'll use `pandas` for structured data, `numpy` for numeric calculations, and Python's built-in `re` (regular expressions) for text parsing. No heavy ML frameworks needed — this lab focuses on evaluation **logic** and **design**.

### Steps
Run the cells below to install dependencies and import libraries.

In [ ]:
%%capture
# Install required packages:
#   - pandas: for creating structured DataFrames of evaluation results
#   - numpy: for numeric calculations (correlation, statistics)
#   - openai: included for reference (used in production LLM-as-judge calls)
!pip install pandas numpy openai

In [ ]:
import pandas as pd               # Data manipulation and reporting
import numpy as np               # Numeric operations (correlation, statistics)
import json                       # Parsing structured data
import re                         # Regular expressions for text parsing / claim extraction
from collections import defaultdict  # Counting patterns in evaluation results

print("✅ All imports successful.")

### Checkpoint ✅
You should see `✅ All imports successful.` with no errors. If you see a `ModuleNotFoundError`, re-run the install cell and restart the kernel.

---

Now that our environment is ready, let's load the data we'll be evaluating. Every evaluation starts with **understanding your ground truth**.

---

## Task 2: Load the Financial Analysis Dataset 📊

### Goal
Load a synthetic dataset of quarterly earnings analysis summaries. Each entry contains the **source financial data** (the ground truth numbers), the **LLM-generated summary**, and metadata about what the summary should cover.

### Why This Dataset?
We need a dataset where we **know the ground truth** so we can measure exactly where the LLM went wrong. In real evaluation work, this "source of truth" might be a database, a knowledge base, or human-written reference answers. Here, the source data acts as our oracle.

> 💡 **Key Insight:** Evaluation is only as good as your ground truth. If you don't know what the correct answer is, you can't measure whether the LLM got it right.

### Dataset Schema
- **company**: Company name
- **quarter**: Reporting quarter
- **source_data**: Raw financial figures (revenue, expenses, margins, etc.) — this is our **ground truth**
- **generated_summary**: The LLM-generated financial analysis — this is **what we're evaluating**
- **key_facts**: List of facts that should appear in a correct summary
- **numeric_claims**: Specific numeric assertions that can be verified against source data
- **human_quality_score**: A human expert rating (1–5) for reference

### Steps
Run the cell below to create the synthetic financial dataset.

In [ ]:
# ──────────────────────────────────────────────────────────────
# Synthetic financial analysis evaluation dataset
# Each entry represents one quarterly summary generated by an LLM.
#
# STRUCTURE OF EACH ENTRY:
#   - source_data: The ACTUAL numbers (our ground truth)
#   - generated_summary: What the LLM wrote (what we're checking)
#   - key_facts: Facts that SHOULD appear (for completeness scoring)
#   - numeric_claims: Specific numbers to verify (stated vs actual)
#   - human_quality_score: Expert rating for comparison to our automated scores
# ──────────────────────────────────────────────────────────────

financial_dataset = [
    {
        "id": 1,
        "company": "TechFlow Inc.",
        "quarter": "Q3 2025",
        "source_data": {
            "revenue": 142_000_000,
            "revenue_prev_quarter": 128_000_000,
            "revenue_prev_year": 118_000_000,
            "operating_expenses": 98_000_000,
            "net_income": 31_000_000,
            "net_income_prev_quarter": 27_000_000,
            "gross_margin_pct": 45.2,
            "operating_margin_pct": 31.0,
            "eps": 2.15,
            "eps_prev_quarter": 1.88,
            "total_debt": 85_000_000,
            "cash_on_hand": 210_000_000,
            "headcount": 3_200,
            "guidance_next_quarter_revenue_low": 148_000_000,
            "guidance_next_quarter_revenue_high": 155_000_000,
            "notable_events": ["Acquired DataSync Corp for $45M", "Launched CloudEdge platform"]
        },
        # This summary is HIGH QUALITY — accurate, complete, well-framed
        "generated_summary": "TechFlow Inc. reported strong Q3 2025 results with revenue of $142M, up 10.9% quarter-over-quarter and 20.3% year-over-year. Net income reached $31M with EPS of $2.15, representing 14.4% growth over Q2. The company maintains a healthy balance sheet with $210M in cash against $85M in total debt. Operating margin was 31.0% with gross margins at 45.2%. During the quarter, TechFlow acquired DataSync Corp for $45M and launched the CloudEdge platform. Management guided Q4 revenue to $148M-$155M, signaling continued momentum.",
        "key_facts": [
            "Revenue of $142M",
            "Quarter-over-quarter revenue growth",
            "Year-over-year revenue growth",
            "Net income of $31M",
            "EPS of $2.15",
            "Cash position of $210M",
            "Debt position of $85M",
            "Acquisition of DataSync Corp",
            "Launch of CloudEdge platform",
            "Q4 revenue guidance range"
        ],
        "numeric_claims": {
            "revenue": {"stated": 142_000_000, "actual": 142_000_000},
            "qoq_growth_pct": {"stated": 10.9, "actual": 10.94},
            "yoy_growth_pct": {"stated": 20.3, "actual": 20.34},
            "eps": {"stated": 2.15, "actual": 2.15},
            "eps_growth_pct": {"stated": 14.4, "actual": 14.36},
            "net_income": {"stated": 31_000_000, "actual": 31_000_000}
        },
        "human_quality_score": 5  # Expert says: excellent summary
    },
    {
        "id": 2,
        "company": "GreenLeaf Energy",
        "quarter": "Q3 2025",
        "source_data": {
            "revenue": 89_000_000,
            "revenue_prev_quarter": 94_000_000,
            "revenue_prev_year": 82_000_000,
            "operating_expenses": 78_000_000,
            "net_income": 4_500_000,
            "net_income_prev_quarter": 11_000_000,
            "gross_margin_pct": 28.1,
            "operating_margin_pct": 12.4,
            "eps": 0.38,
            "eps_prev_quarter": 0.92,
            "total_debt": 220_000_000,
            "cash_on_hand": 45_000_000,
            "headcount": 1_800,
            "guidance_next_quarter_revenue_low": 91_000_000,
            "guidance_next_quarter_revenue_high": 97_000_000,
            "notable_events": ["Debt covenant waiver obtained from lenders", "CEO transition announced"]
        },
        # ⚠️ This summary is MISLEADING — everything stated is true,
        # but it OMITS critical negative information (declining revenue,
        # plummeting income, debt covenant waiver, CEO leaving)
        "generated_summary": "GreenLeaf Energy posted Q3 2025 revenue of $89M, growing 8.5% year-over-year. Operating margin remained healthy at 12.4%. EPS came in at $0.38. The company has $45M in cash and continues to invest in renewable infrastructure expansion. Management expects Q4 revenue between $91M and $97M, reflecting confidence in their growth strategy.",
        "key_facts": [
            "Revenue of $89M",
            "Revenue declined quarter-over-quarter",
            "Year-over-year revenue growth",
            "Significant net income decline",
            "EPS of $0.38 (sharp drop from $0.92)",
            "High debt level of $220M",
            "Low cash position of $45M relative to debt",
            "Debt covenant waiver obtained",
            "CEO transition announced",
            "Q4 revenue guidance range"
        ],
        "numeric_claims": {
            "revenue": {"stated": 89_000_000, "actual": 89_000_000},
            "yoy_growth_pct": {"stated": 8.5, "actual": 8.54},
            "operating_margin_pct": {"stated": 12.4, "actual": 12.4},
            "eps": {"stated": 0.38, "actual": 0.38}
        },
        "human_quality_score": 2  # Expert says: dangerously misleading
    },
    {
        "id": 3,
        "company": "MedVault Health",
        "quarter": "Q3 2025",
        "source_data": {
            "revenue": 210_000_000,
            "revenue_prev_quarter": 198_000_000,
            "revenue_prev_year": 175_000_000,
            "operating_expenses": 165_000_000,
            "net_income": 33_000_000,
            "net_income_prev_quarter": 29_000_000,
            "gross_margin_pct": 52.8,
            "operating_margin_pct": 21.4,
            "eps": 1.72,
            "eps_prev_quarter": 1.51,
            "total_debt": 150_000_000,
            "cash_on_hand": 320_000_000,
            "headcount": 5_400,
            "guidance_next_quarter_revenue_low": 218_000_000,
            "guidance_next_quarter_revenue_high": 225_000_000,
            "notable_events": ["FDA approval for DiagnostiQ AI platform", "Expanded into 3 new European markets"]
        },
        # ⚠️ This summary has a NUMERIC ERROR — states 25% YoY growth
        # but the actual calculation is (210-175)/175 = 20.0%
        "generated_summary": "MedVault Health delivered solid Q3 2025 results. Revenue hit $210M, up 6.1% from Q2 and 25% year-over-year — a strong acceleration driven by the new DiagnostiQ AI platform which received FDA approval during the quarter. Net income was $33M with EPS of $1.72. Gross margins expanded to 52.8%, among the best in the health-tech sector. The company expanded into 3 new European markets and holds a fortress balance sheet with $320M in cash versus $150M in debt. Operating expenses rose to $165M reflecting investment in European expansion. Q4 guidance of $218M-$225M suggests management expects mid-single-digit sequential growth.",
        "key_facts": [
            "Revenue of $210M",
            "Quarter-over-quarter revenue growth",
            "Year-over-year revenue growth",
            "Net income of $33M",
            "EPS of $1.72",
            "Gross margin of 52.8%",
            "FDA approval for DiagnostiQ AI",
            "European market expansion",
            "Cash position of $320M",
            "Debt of $150M",
            "Q4 revenue guidance range"
        ],
        "numeric_claims": {
            "revenue": {"stated": 210_000_000, "actual": 210_000_000},
            "qoq_growth_pct": {"stated": 6.1, "actual": 6.06},
            "yoy_growth_pct": {"stated": 25.0, "actual": 20.0},  # <-- THE ERROR: 25% vs 20%
            "eps": {"stated": 1.72, "actual": 1.72},
            "gross_margin_pct": {"stated": 52.8, "actual": 52.8},
            "net_income": {"stated": 33_000_000, "actual": 33_000_000}
        },
        "human_quality_score": 3  # Expert says: good but has a numeric error
    },
    {
        "id": 4,
        "company": "UrbanLogix",
        "quarter": "Q3 2025",
        "source_data": {
            "revenue": 67_000_000,
            "revenue_prev_quarter": 71_000_000,
            "revenue_prev_year": 63_000_000,
            "operating_expenses": 72_000_000,
            "net_income": -8_500_000,
            "net_income_prev_quarter": -3_200_000,
            "gross_margin_pct": 18.7,
            "operating_margin_pct": -7.5,
            "eps": -0.62,
            "eps_prev_quarter": -0.23,
            "total_debt": 180_000_000,
            "cash_on_hand": 52_000_000,
            "headcount": 2_100,
            "guidance_next_quarter_revenue_low": 70_000_000,
            "guidance_next_quarter_revenue_high": 75_000_000,
            "notable_events": ["Restructuring plan announced — 15% workforce reduction", "Entered partnership with CityGrid for smart delivery"]
        },
        # ⚠️ This summary SPINS negative news positively —
        # calls widening losses "strategic investments" and omits the layoffs
        "generated_summary": "UrbanLogix reported Q3 2025 revenue of $67M, up 6.3% year-over-year, demonstrating resilience in the logistics space. While operating expenses of $72M exceeded revenue, the company's strategic investments are positioning it for long-term growth. EPS was -$0.62. The company announced an exciting partnership with CityGrid for smart delivery solutions and has $52M in cash to fund operations. Management guided Q4 revenue to $70M-$75M.",
        "key_facts": [
            "Revenue of $67M",
            "Revenue declined quarter-over-quarter",
            "Year-over-year revenue growth",
            "Net loss of $8.5M (widening losses)",
            "Negative operating margin of -7.5%",
            "EPS of -$0.62 (worsening from -$0.23)",
            "High debt of $180M vs $52M cash",
            "15% workforce restructuring",
            "CityGrid partnership",
            "Q4 revenue guidance range"
        ],
        "numeric_claims": {
            "revenue": {"stated": 67_000_000, "actual": 67_000_000},
            "yoy_growth_pct": {"stated": 6.3, "actual": 6.35},
            "operating_expenses": {"stated": 72_000_000, "actual": 72_000_000},
            "eps": {"stated": -0.62, "actual": -0.62}
        },
        "human_quality_score": 2  # Expert says: misleading positive spin
    },
    {
        "id": 5,
        "company": "NexaPay Financial",
        "quarter": "Q3 2025",
        "source_data": {
            "revenue": 325_000_000,
            "revenue_prev_quarter": 310_000_000,
            "revenue_prev_year": 280_000_000,
            "operating_expenses": 245_000_000,
            "net_income": 58_000_000,
            "net_income_prev_quarter": 52_000_000,
            "gross_margin_pct": 61.3,
            "operating_margin_pct": 24.6,
            "eps": 3.42,
            "eps_prev_quarter": 3.07,
            "total_debt": 120_000_000,
            "cash_on_hand": 480_000_000,
            "headcount": 8_200,
            "guidance_next_quarter_revenue_low": 335_000_000,
            "guidance_next_quarter_revenue_high": 345_000_000,
            "notable_events": ["Launched instant cross-border payments in 12 new countries", "Regulatory approval for crypto custody services"]
        },
        # ⚠️ This summary has FABRICATED details — says 15 countries
        # (actual: 12) and EPS growth of 22% (actual: ~11.4%)
        "generated_summary": "NexaPay Financial posted exceptional Q3 2025 results with revenue of $325M, up 4.8% sequentially and 16.1% year-over-year. Net income surged to $58M, driving EPS to $3.42 — a 22% increase versus Q2. Gross margins of 61.3% and operating margins of 24.6% reflect excellent unit economics. The company launched instant cross-border payments in 15 new countries and received regulatory approval for crypto custody services. With $480M in cash and only $120M in debt, NexaPay has significant capacity for M&A. Q4 guidance of $335M-$345M represents 3-6% sequential growth. One risk to watch: operating expenses of $245M grew faster than revenue on a percentage basis.",
        "key_facts": [
            "Revenue of $325M",
            "Quarter-over-quarter revenue growth",
            "Year-over-year revenue growth",
            "Net income of $58M",
            "EPS of $3.42",
            "Gross margin of 61.3%",
            "Operating margin of 24.6%",
            "Cross-border payments expansion",
            "Crypto custody regulatory approval",
            "Strong cash position ($480M vs $120M debt)",
            "Q4 revenue guidance range"
        ],
        "numeric_claims": {
            "revenue": {"stated": 325_000_000, "actual": 325_000_000},
            "qoq_growth_pct": {"stated": 4.8, "actual": 4.84},
            "yoy_growth_pct": {"stated": 16.1, "actual": 16.07},
            "eps": {"stated": 3.42, "actual": 3.42},
            "eps_growth_pct": {"stated": 22.0, "actual": 11.40},  # <-- WRONG: 22% vs 11.4%
            "net_income": {"stated": 58_000_000, "actual": 58_000_000},
            "countries_expanded": {"stated": 15, "actual": 12}  # <-- FABRICATED: 15 vs 12
        },
        "human_quality_score": 3  # Expert says: mostly good but has errors
    },
    {
        "id": 6,
        "company": "SkyBridge Telecom",
        "quarter": "Q3 2025",
        "source_data": {
            "revenue": 195_000_000,
            "revenue_prev_quarter": 192_000_000,
            "revenue_prev_year": 188_000_000,
            "operating_expenses": 170_000_000,
            "net_income": 18_000_000,
            "net_income_prev_quarter": 16_500_000,
            "gross_margin_pct": 38.5,
            "operating_margin_pct": 12.8,
            "eps": 1.05,
            "eps_prev_quarter": 0.96,
            "total_debt": 340_000_000,
            "cash_on_hand": 75_000_000,
            "headcount": 4_600,
            "guidance_next_quarter_revenue_low": 196_000_000,
            "guidance_next_quarter_revenue_high": 202_000_000,
            "notable_events": ["5G network coverage reached 80% of target markets", "Signed 10-year spectrum lease agreement"]
        },
        # This summary is DECENT — accurate but misses the $340M debt position
        "generated_summary": "SkyBridge Telecom delivered steady Q3 2025 results with revenue of $195M, a modest 1.6% increase over Q2 and 3.7% year-over-year growth. Net income of $18M and EPS of $1.05 show continued profitability. The company achieved a major milestone with 5G coverage reaching 80% of target markets and secured a 10-year spectrum lease. Operating margin of 12.8% was stable. Cash on hand stands at $75M. Q4 guidance of $196M–$202M suggests continued steady performance.",
        "key_facts": [
            "Revenue of $195M",
            "Quarter-over-quarter revenue growth",
            "Year-over-year revenue growth",
            "Net income of $18M",
            "EPS of $1.05",
            "5G coverage milestone (80%)",
            "Spectrum lease agreement",
            "Debt position of $340M",
            "Cash position of $75M",
            "Q4 revenue guidance range"
        ],
        "numeric_claims": {
            "revenue": {"stated": 195_000_000, "actual": 195_000_000},
            "qoq_growth_pct": {"stated": 1.6, "actual": 1.56},
            "yoy_growth_pct": {"stated": 3.7, "actual": 3.72},
            "eps": {"stated": 1.05, "actual": 1.05},
            "net_income": {"stated": 18_000_000, "actual": 18_000_000}
        },
        "human_quality_score": 4  # Expert says: solid but missing debt info
    }
]

# ── Quick summary of what we loaded ──
print(f"Loaded {len(financial_dataset)} financial analysis summaries.\n")
for item in financial_dataset:
    src = item["source_data"]
    print(f"  Summary {item['id']}: {item['company']} ({item['quarter']})")
    print(f"    Revenue: ${src['revenue']/1e6:.0f}M | Net Income: ${src['net_income']/1e6:.1f}M | Human Score: {item['human_quality_score']}/5")
    print(f"    Key facts to check: {len(item['key_facts'])} | Numeric claims: {len(item['numeric_claims'])}")
    print()

### Checkpoint ✅
You should see 6 financial analysis summaries loaded, each with:
- Company name and quarter
- Revenue and net income figures
- A human quality score from 1–5
- Counts of key facts and numeric claims to verify

### Expected Output (partial)
```
Loaded 6 financial analysis summaries.

  Summary 1: TechFlow Inc. (Q3 2025)
    Revenue: $142M | Net Income: $31.0M | Human Score: 5/5
    Key facts to check: 10 | Numeric claims: 6
  ...
```

Notice the variation in human quality scores. Summary 1 gets a 5/5, but summaries 2 and 4 only get 2/5. You'll discover why as you evaluate them.

### 🔍 What Just Happened?
We loaded 6 LLM-generated financial summaries, each paired with **ground truth source data**. This is the foundation of our entire evaluation pipeline — without source data to compare against, we can't distinguish accurate statements from hallucinations. Notice that human experts have already scored these summaries from 1–5. Our goal is to build automated evaluation that **agrees with and explains** those human judgments.

---

## Task 3: 🔍 Think First — What Makes a Financial Summary Trustworthy?

### Goal
Before building any evaluation logic, pause and reason about what makes a financial summary **trustworthy** vs **dangerous**. In finance, bad summaries don't just waste time — they can cause **real financial losses** and **regulatory violations**.

> 🎯 **Why This Matters:** You wouldn't build a bridge without understanding what forces it needs to withstand. Similarly, before building an evaluator, you need to understand **what kinds of errors** you're trying to catch. This task builds your mental model.

### Context
Look at this generated summary for GreenLeaf Energy (Summary 2):

> "GreenLeaf Energy posted Q3 2025 revenue of $89M, growing 8.5% year-over-year. Operating margin remained healthy at 12.4%. EPS came in at $0.38. The company has $45M in cash and continues to invest in renewable infrastructure expansion. Management expects Q4 revenue between $91M and $97M, reflecting confidence in their growth strategy."

Now look at the **source data** — you can inspect it in the dataset above (Summary 2). Notice in particular:
- Revenue went **down** from Q2 ($94M → $89M) — but the summary doesn't mention this
- Net income plummeted from $11M to $4.5M — not mentioned
- EPS crashed from $0.92 to $0.38 — presented without context
- The company has $220M in debt against $45M cash — debt not mentioned
- A **debt covenant waiver** was obtained — a critical risk signal, omitted
- The **CEO is leaving** — also omitted

> 💡 **Key Insight:** The summary is factually accurate in what it states, but it paints a **misleadingly positive picture** by omitting critical negative information. This is one of the hardest errors to catch — and one of the most dangerous.

### Questions to Reason About

Spend **3–4 minutes** thinking about these questions. Write your answers in the cell below.

1. **Types of errors:** What are the different ways a financial summary can be wrong? Think beyond just "incorrect numbers."

2. **Severity ranking:** Rank these errors from most to least dangerous in a financial context:
   - Rounding a growth rate (10.94% → 10.9%)
   - Omitting a debt covenant waiver
   - Stating revenue grew when it declined QoQ
   - Getting the revenue number slightly wrong ($142M reported as $143M)
   - Using optimistic framing for declining metrics

3. **Omission vs fabrication:** Which is more dangerous in financial reporting — leaving out critical information or stating something false? Why?

4. **Evaluation dimensions:** What distinct dimensions would you evaluate a financial summary on? Try to list at least 4 different aspects.

> ⚠️ **Common Mistake:** Many people assume evaluation = "are the facts correct?" But as GreenLeaf shows, a summary can be 100% factually correct and still be dangerously misleading. A good evaluator checks **multiple dimensions**.

> **This is a challenge section.** There are no answers provided here. Form your own reasoning — it will shape how you approach the evaluation pipeline.

### ✍️ Your Reflections

**Write your answers here** (double-click to edit this cell):

1. **Types of errors:**

   *Your answer...*

2. **Severity ranking:**

   *Your answer...*

3. **Omission vs fabrication:**

   *Your answer...*

4. **Evaluation dimensions:**

   *Your answer...*

---
*Keep this cell — your reasoning here will inform how you approach the evaluation tasks below.*

---

Now that you've thought about what can go wrong, let's build the first layer of our evaluation pipeline: **factuality scoring**. This is where we check whether what the LLM said is actually true.

## Task 4: Implement Factuality Scoring (Claim Extraction + Verification) 🔍

### Goal
Build a factuality scoring pipeline that:
1. Extracts individual claims from a generated summary
2. Verifies each claim against the source data
3. Classifies claims as **supported**, **unsupported**, **contradicted**, or **fabricated**
4. Produces a factuality score

### Why Does Factuality Scoring Matter?

LLM-generated text can sound **perfectly fluent and confident** while being completely wrong. This is the hallucination problem — and it's especially dangerous because:

- Humans tend to trust confident, well-written text
- LLMs don't "know" they're making things up
- In finance, a single wrong number can lead to million-dollar trade decisions

> 💡 **Key Insight:** Fluent text can be completely fabricated — evaluation must go deeper than "does it read well?"

### Types of Hallucination

```
┌─────────────────────────────────────────────────────────────┐
│                  HALLUCINATION TYPES                        │
├─────────────────┬───────────────────┬───────────────────────┤
│  FABRICATION    │   DISTORTION      │    CONFLATION         │
│                 │                   │                       │
│  Making up      │  Taking a real    │  Mixing up details   │
│  facts that     │  fact and getting │  from different      │
│  don't exist    │  the details      │  sources or time     │
│  at all         │  wrong            │  periods             │
│                 │                   │                       │
│  Example:       │  Example:         │  Example:            │
│  "15 countries" │  "25% YoY growth" │  Using Q2 numbers    │
│  (actual: 12)   │  (actual: 20%)    │  in a Q3 summary     │
└─────────────────┴───────────────────┴───────────────────────┘
```

### Claim Categories
- **Supported**: Claim is directly verifiable from source data and is correct
- **Unsupported**: Claim cannot be verified from source (may or may not be true)
- **Contradicted**: Claim directly conflicts with source data
- **Fabricated**: Claim introduces specific details not present in source data

### Steps
1. Review the claim extraction function
2. Run factuality scoring on all summaries
3. Analyze the results to identify which summaries have factual issues

In [ ]:
# ──────────────────────────────────────────────────────────────
# STEP 1: CLAIM EXTRACTION
# Break a summary into individual, checkable claims.
# Think of this as the "decompose" step — we can't verify a whole
# paragraph at once, but we CAN verify individual statements.
# ──────────────────────────────────────────────────────────────

def extract_claims(summary_text):
    """
    Extract individual factual claims from a financial summary.
    Splits on sentence boundaries and filters out pure opinion/framing.

    In production, you would use an LLM for claim extraction.
    Here we use a rule-based approach for reproducibility.
    """
    # Split into sentences at period/exclamation/question marks
    sentences = re.split(r'(?<=[.!?])\s+', summary_text.strip())
    claims = []
    for sent in sentences:
        # Split compound sentences (e.g., "Revenue grew and margins expanded")
        # so each part becomes a separate, verifiable claim
        parts = re.split(r'\s+and\s+|\s+while\s+', sent)
        for part in parts:
            part = part.strip()
            if len(part) > 15:  # Skip very short fragments that aren't real claims
                claims.append(part)
    return claims


# ──────────────────────────────────────────────────────────────
# STEP 2: CLAIM VERIFICATION
# For each extracted claim, check whether it's supported by
# the source data. This is the core of factuality scoring.
# ──────────────────────────────────────────────────────────────

def verify_claim_against_source(claim, source_data, numeric_claims):
    """
    Verify a single claim against source data.
    Returns: (status, explanation)

    Status: 'supported', 'unsupported', 'contradicted', 'fabricated'
    """
    claim_lower = claim.lower()

    # ── Priority 1: Check numeric claims ──
    # Numeric claims are the most verifiable — we can precisely compare
    # stated values to actual values with a tolerance for rounding
    for metric, values in numeric_claims.items():
        stated = values["stated"]
        actual = values["actual"]

        # Check if this claim mentions this metric
        stated_str = str(stated)
        if isinstance(stated, float):
            stated_str = f"{stated}"

        # Look for the stated value in the claim text
        if _claim_references_metric(claim_lower, metric, stated):
            # Compare stated value to actual with tolerance
            if isinstance(stated, (int, float)) and isinstance(actual, (int, float)):
                tolerance = max(abs(actual) * 0.02, 0.1)  # 2% tolerance or 0.1 absolute
                if abs(stated - actual) <= tolerance:
                    return "supported", f"{metric}: stated {stated} matches actual {actual}"
                else:
                    # The claim CONTRADICTS the source — this is a factual error
                    return "contradicted", f"{metric}: stated {stated} but actual is {actual} (difference: {abs(stated - actual):.2f})"

    # ── Priority 2: Check notable events ──
    # See if the claim mentions a real event from the source data
    for event in source_data.get("notable_events", []):
        event_keywords = set(event.lower().split())
        claim_words = set(claim_lower.split())
        overlap = event_keywords & claim_words
        if len(overlap) >= 3:  # At least 3 words in common = likely match
            return "supported", f"Matches notable event: {event}"

    # ── Priority 3: Check key financial figures (revenue, income, cash, debt) ──
    revenue = source_data.get("revenue", 0)
    if _mentions_revenue(claim_lower) and _mentions_amount(claim_lower, revenue):
        return "supported", "Revenue figure matches source"

    net_income = source_data.get("net_income", 0)
    if _mentions_net_income(claim_lower) and _mentions_amount(claim_lower, net_income):
        return "supported", "Net income figure matches source"

    cash = source_data.get("cash_on_hand", 0)
    if "cash" in claim_lower and _mentions_amount(claim_lower, cash):
        return "supported", "Cash figure matches source"

    debt = source_data.get("total_debt", 0)
    if "debt" in claim_lower and _mentions_amount(claim_lower, debt):
        return "supported", "Debt figure matches source"

    # ── Priority 4: Flag claims with unverified numbers ──
    # If a claim contains numbers we can't verify, mark as unsupported
    numbers_in_claim = re.findall(r'\$?[\d,]+\.?\d*[MB]?', claim)
    if numbers_in_claim:
        return "unsupported", f"Contains numbers {numbers_in_claim} - not directly verified"

    # ── Priority 5: Flag subjective framing ──
    # Words like "strong", "exceptional" etc. are opinions, not verifiable facts
    framing_indicators = ["strong", "solid", "exceptional", "resilience", "exciting",
                          "healthy", "significant", "continued", "steady", "fortress"]
    if any(word in claim_lower for word in framing_indicators):
        return "unsupported", "Subjective framing - cannot verify against data"

    # Fallback: can't verify this claim either way
    return "unsupported", "Claim not directly verifiable from source data"


# ── Helper functions for matching claims to source data ──

def _claim_references_metric(claim_lower, metric, value):
    """Check if a claim references a particular metric by looking for
    both the metric keyword AND the stated value in the claim text."""
    metric_keywords = {
        "revenue": ["revenue"],
        "qoq_growth_pct": ["quarter-over-quarter", "sequentially", "from q", "over q2", "q2"],
        "yoy_growth_pct": ["year-over-year", "yoy"],
        "eps": ["eps", "earnings per share"],
        "eps_growth_pct": ["eps", "earnings per share"],
        "operating_margin_pct": ["operating margin"],
        "gross_margin_pct": ["gross margin"],
        "net_income": ["net income"],
        "operating_expenses": ["operating expense"],
        "countries_expanded": ["countries", "country"]
    }
    keywords = metric_keywords.get(metric, [metric.replace("_", " ")])
    value_str = str(value) if isinstance(value, int) else f"{value}"
    # Both conditions must be true: claim mentions the metric AND the value
    return any(kw in claim_lower for kw in keywords) and value_str in claim_lower.replace(",", "")


def _mentions_revenue(text):
    """Does this text talk about revenue?"""
    return "revenue" in text


def _mentions_net_income(text):
    """Does this text talk about net income or net loss?"""
    return "net income" in text or "net loss" in text


def _mentions_amount(text, amount):
    """Check if text mentions an amount in $M format (e.g., $142M, $31.0M)."""
    millions = amount / 1e6
    patterns = [f"${millions:.0f}m", f"${millions:.1f}m", f"${int(millions)}m"]
    return any(p in text.replace(" ", "").lower() for p in patterns)


print("✅ Factuality scoring functions defined.")
print("   - extract_claims(): Breaks summaries into individual checkable claims")
print("   - verify_claim_against_source(): Checks each claim against ground truth")

In [ ]:
# ──────────────────────────────────────────────────────────────
# Run factuality scoring on ALL summaries
# For each summary: extract claims → verify each → compute score
# ──────────────────────────────────────────────────────────────
print("=" * 80)
print("FACTUALITY SCORING RESULTS")
print("=" * 80)

factuality_results = []

for item in financial_dataset:
    # Step 1: Break the summary into individual claims
    claims = extract_claims(item["generated_summary"])

    # Step 2: Verify each claim against source data
    results = []
    for claim in claims:
        status, explanation = verify_claim_against_source(
            claim, item["source_data"], item["numeric_claims"]
        )
        results.append({"claim": claim, "status": status, "explanation": explanation})

    # Step 3: Count results by category
    supported = sum(1 for r in results if r["status"] == "supported")
    contradicted = sum(1 for r in results if r["status"] == "contradicted")
    fabricated = sum(1 for r in results if r["status"] == "fabricated")
    unsupported = sum(1 for r in results if r["status"] == "unsupported")
    total = len(results)

    # Step 4: Compute factuality score
    # Contradicted and fabricated claims are penalized DOUBLE (weight of -2)
    # because false information is worse than unverifiable information
    factuality_score = (supported - contradicted * 2 - fabricated * 2) / total if total > 0 else 0

    factuality_results.append({
        "id": item["id"],
        "company": item["company"],
        "total_claims": total,
        "supported": supported,
        "contradicted": contradicted,
        "fabricated": fabricated,
        "unsupported": unsupported,
        "factuality_score": factuality_score,
        "details": results
    })

    # Step 5: Print results for this summary
    print(f"\n{'─' * 60}")
    print(f"Summary {item['id']}: {item['company']} ({item['quarter']})")
    print(f"  Claims extracted: {total}")
    print(f"  ✅ Supported:    {supported}")
    print(f"  ❌ Contradicted: {contradicted}")
    print(f"  🔴 Fabricated:   {fabricated}")
    print(f"  ❓ Unsupported:  {unsupported}")
    print(f"  📊 Factuality Score: {factuality_score:.3f}")

    # Highlight any problematic claims — these are the errors we caught!
    issues = [r for r in results if r["status"] in ("contradicted", "fabricated")]
    if issues:
        print(f"\n  ⚠️  ISSUES FOUND:")
        for issue in issues:
            print(f"    [{issue['status'].upper()}] \"{issue['claim'][:80]}...\"")
            print(f"      → {issue['explanation']}")

### Checkpoint ✅
You should see factuality scoring results for all 6 summaries. What the verified run shows:
- **TechFlow, MedVault, SkyBridge**: factuality ≈ 0.63–0.67. Most claims supported, a few "unsupported" (claims the keyword verifier can't check either way)
- **NexaPay (Summary 5)**: the lowest score (≈ 0.36) and a **CONTRADICTED** claim: "15 new countries" vs. 12 in the source
- **MedVault (Summary 3)**: ⚠️ its "25% year-over-year" claim is **not** flagged here, even though the true figure is 20%. The keyword-based verifier doesn't map that sentence to the growth metric. Hold that thought: the **numeric layer in Task 6 catches it**

### Expected Output (partial)
```
Summary 5: NexaPay Financial (Q3 2025)
  ❌ Contradicted: 1
  📊 Factuality Score: 0.364
  ⚠️  ISSUES FOUND:
    [CONTRADICTED] "The company launched instant cross-border payments in 15 new countries..."
      → countries_expanded: stated 15 but actual is 12
```

### 🔍 What Just Happened?
We built the **first layer** of our evaluation pipeline: factuality scoring. We decomposed each summary into individual claims, then verified each one against source data. This caught NexaPay's **fabricated country count**. Notice what it *didn't* catch: MedVault's inflated growth rate slipped through a keyword-based verifier (a **false negative** worth remembering for the Stretch Goal), and GreenLeaf scored reasonably well because everything it stated was technically true. The problem is what it **left out**. That's what completeness scoring addresses next.

---

Factuality scoring caught summaries with **wrong** information. But what about summaries that are correct yet **incomplete**? That's the next dimension we need to evaluate.

## Task 5: 🧠 Your Turn — Design Completeness Criteria for Financial Summaries

### Goal
Design a **completeness evaluation framework** for financial summaries. Completeness measures whether the summary covers all the information it **should** cover — especially critical negative information.

### Why Completeness Matters

> 💡 **Key Insight:** A summary can be 100% factually correct and still be dangerously misleading. Omission is a form of deception — especially in regulated domains like finance.

### The Completeness vs Conciseness Tradeoff

This is one of the central tensions in generative output evaluation:

```
           Completeness vs Conciseness Tradeoff
           ─────────────────────────────────────

  High  │                        ╱  Ideal Zone
  Comp- │  Wall of text ──────╱    (covers all material
  lete- │  (every detail,   ╱       facts concisely)
  ness  │   unreadable)   ╱
        │               ╱
        │             ╱
        │   ⚠️      ╱  Missing critical info
        │  (some  ╱    (looks clean but
        │  facts)╱      omits key risks)
  Low   │      ╱
        └──────────────────────────────
          Low                     High
                  Conciseness
```

> ⚠️ **Common Mistake:** Don't confuse conciseness with incompleteness. A concise summary covers all **material** facts efficiently. An incomplete summary **skips** material facts to look concise.

### Context
You saw in Task 3 that the GreenLeaf Energy summary (Summary 2) omitted:
- Quarter-over-quarter revenue decline
- Net income plummeting 59%
- EPS dropping from $0.92 to $0.38
- $220M debt position
- Debt covenant waiver (a serious risk signal)
- CEO transition

This is a completeness failure, not a factuality failure — everything stated was true, but critical information was missing.

### 🧠 Your Challenge

Design completeness criteria by answering these questions in the markdown cell below:

1. **Required categories:** What categories of information must ALWAYS be present in a quarterly financial summary? List at least 5 categories.

2. **Critical omissions:** Define what counts as a "critical" omission vs a "minor" omission. Give examples of each in a financial context.

3. **Negative information rule:** Propose a specific rule for how negative trends (declining revenue, widening losses, etc.) should be weighted in completeness scoring. Should omitting bad news be penalized more than omitting good news?

4. **Scoring formula:** Propose a completeness scoring formula. How would you calculate a score from 0 to 1?

**Hint:** Think about categories like: financial performance, risk factors, guidance, notable events, balance sheet health. Consider whether all categories should be weighted equally.

> **Do NOT look ahead for the answer.** Design your own criteria first. You'll implement a completeness scorer in the next task using your criteria.

### ✍️ Your Completeness Criteria Design

**Write your design here** (double-click to edit this cell):

1. **Required categories:**

   *Your answer...*

2. **Critical vs minor omissions:**

   *Your answer...*

3. **Negative information rule:**

   *Your answer...*

4. **Scoring formula:**

   *Your answer...*

---
*You will implement this in the next task.*

In [ ]:
# ──────────────────────────────────────────────────────────────
# COMPLETENESS SCORER
# Now implement a completeness scorer.
# This reference implementation uses WEIGHTED categories:
#   - Critical facts (risk events, negative trends) get weight 2.0
#   - Standard facts get weight 1.0
# Compare it to your design above — what's similar? What's different?
# ──────────────────────────────────────────────────────────────

def score_completeness(generated_summary, key_facts, source_data):
    """
    Score completeness of a financial summary based on coverage of key facts.

    The key idea: not all facts are equally important. Omitting a debt
    covenant waiver (risk event) is much worse than omitting a headcount
    figure. So we WEIGHT critical facts more heavily.

    Returns:
      - overall_score (0-1)
      - category_scores dict
      - missing_facts list
    """
    summary_lower = generated_summary.lower()

    # ── Define which keywords make a fact "critical" ──
    # These words indicate risk events, negative trends, or material concerns
    # that a portfolio manager MUST know about
    critical_keywords = [
        "debt", "covenant", "restructuring", "ceo", "decline", "loss",
        "widening", "negative", "worsening", "reduction", "workforce"
    ]

    covered_facts = []
    missing_facts = []
    total_weight = 0
    covered_weight = 0

    for fact in key_facts:
        fact_lower = fact.lower()

        # Determine if this is a critical fact (weight 2x) or standard (weight 1x)
        is_critical = any(kw in fact_lower for kw in critical_keywords)
        weight = 2.0 if is_critical else 1.0
        total_weight += weight

        # ── Check if the summary covers this fact ──
        # Use keyword overlap: extract meaningful words from the fact,
        # then see how many of those words appear in the summary
        fact_keywords = set(re.findall(r'\b\w+\b', fact_lower))
        # Remove common words that don't carry meaning
        fact_keywords -= {"of", "the", "a", "an", "in", "for", "to", "and", "or", "is", "was"}
        summary_words = set(re.findall(r'\b\w+\b', summary_lower))

        # Require at least 50% keyword overlap to count as "covered"
        overlap = fact_keywords & summary_words
        coverage_ratio = len(overlap) / len(fact_keywords) if fact_keywords else 0

        if coverage_ratio >= 0.5:
            covered_facts.append({"fact": fact, "weight": weight, "critical": is_critical})
            covered_weight += weight
        else:
            missing_facts.append({"fact": fact, "weight": weight, "critical": is_critical})

    # Overall score = weighted coverage (0 to 1)
    # A summary that covers all standard facts but misses critical ones
    # will score lower than one that covers everything
    overall_score = covered_weight / total_weight if total_weight > 0 else 0

    return {
        "overall_score": overall_score,
        "facts_covered": len(covered_facts),
        "facts_missing": len(missing_facts),
        "total_facts": len(key_facts),
        "covered_details": covered_facts,
        "missing_details": missing_facts,
        "critical_missing": [f for f in missing_facts if f["critical"]]
    }


print("✅ Completeness scoring function defined.")
print("   Critical facts (risks, negative trends) are weighted 2x.")

In [ ]:
# ──────────────────────────────────────────────────────────────
# Run completeness scoring on ALL summaries
# This will reveal which summaries are hiding bad news
# ──────────────────────────────────────────────────────────────
print("=" * 80)
print("COMPLETENESS SCORING RESULTS")
print("=" * 80)

completeness_results = []

for item in financial_dataset:
    result = score_completeness(
        item["generated_summary"],
        item["key_facts"],
        item["source_data"]
    )
    result["id"] = item["id"]
    result["company"] = item["company"]
    completeness_results.append(result)

    print(f"\n{'─' * 60}")
    print(f"Summary {item['id']}: {item['company']}")
    print(f"  Coverage: {result['facts_covered']}/{result['total_facts']} facts covered")
    print(f"  📊 Completeness Score: {result['overall_score']:.3f}")

    # 🚨 Flag critical omissions — these are the most dangerous gaps
    if result["critical_missing"]:
        print(f"\n  🚨 CRITICAL OMISSIONS ({len(result['critical_missing'])}):")
        for m in result["critical_missing"]:
            print(f"    • {m['fact']}  [weight: {m['weight']}]")

    # ⚠️ Also show non-critical missing facts for completeness
    if result["missing_details"]:
        non_critical = [f for f in result["missing_details"] if not f["critical"]]
        if non_critical:
            print(f"\n  ⚠️  Other missing facts ({len(non_critical)}):")
            for m in non_critical:
                print(f"    • {m['fact']}")

# ── Summary table for quick comparison ──
print(f"\n{'=' * 80}")
print("SUMMARY")
print(f"{'=' * 80}")
for r in completeness_results:
    critical_count = len(r["critical_missing"])
    flag = " 🚨" if critical_count > 0 else " ✅"
    print(f"  {r['company']:25s} | Score: {r['overall_score']:.3f} | Missing: {r['facts_missing']:2d} | Critical: {critical_count}{flag}")

### Checkpoint ✅
You should see completeness scores for all 6 summaries. Key observations:
- **GreenLeaf Energy (Summary 2)** should have **critical omissions** — debt covenant waiver, CEO transition, QoQ decline
- **UrbanLogix (Summary 4)** should also flag critical omissions — 15% workforce restructuring, worsening losses
- **TechFlow (Summary 1)** should have the highest completeness score
- Summaries that omit negative information are penalized more heavily due to the critical weighting

### Expected Output (partial)
```
  GreenLeaf Energy           | Score: 0.xxx | Missing:  X | Critical: X 🚨
  UrbanLogix                 | Score: 0.xxx | Missing:  X | Critical: X 🚨
```

### 🔍 What Just Happened?
We built the **second layer** of our pipeline: completeness scoring. This is what caught GreenLeaf Energy — a summary that passed factuality checks but failed completeness because it omitted critical risk information. The key design choice was **weighting critical facts 2x** — omitting a debt covenant waiver is much more serious than omitting a headcount figure. Now we have two complementary lenses: "Is what it says true?" (factuality) and "Does it say everything it should?" (completeness). Next up: checking the math.

---

We've checked *what* the summary says (factuality) and *what it leaves out* (completeness). Now let's zoom in on the **numbers** — because LLMs are notoriously bad at math.

## Task 6: Implement Numeric Reasoning Checks 🔢

### Goal
Build an automated numeric verification pipeline that extracts numeric claims from summaries and verifies them against source data — catching calculation errors, rounding issues, and outright numeric fabrications.

### Why a Dedicated Numeric Check?
Our factuality scorer already catches some numeric issues, but this task goes deeper:

> 🎯 **Why This Matters:** LLMs don't actually "calculate" — they predict what numbers look plausible. That's why an LLM might write "25% year-over-year growth" when the real number is 20%. It sounds right, but it's a **hallucinated calculation**. In finance, even small numeric errors compound into wrong projections and bad trade decisions.

Numeric errors in financial summaries are particularly dangerous because:
- They can directly influence investment decisions
- They may violate financial reporting accuracy requirements
- LLMs generate "plausible-looking" numbers that are hard to spot-check
- Even small errors compound (wrong growth rate → wrong revenue projections → wrong valuation)

> ⚠️ **Common Mistake:** Treating all numeric errors equally. Rounding 10.94% to 10.9% is acceptable. Getting 25% instead of 20% is a material misstatement. Severity classification matters!

### Steps
1. Review the numeric verification function
2. Run it on all summaries
3. Classify errors by severity

In [ ]:
# ──────────────────────────────────────────────────────────────
# NUMERIC REASONING VERIFICATION
# For each numeric claim (growth rates, margins, dollar amounts),
# compare stated value to actual value and classify the error severity.
#
# Severity levels:
#   - exact:       Values match precisely or within rounding (< 0.5% off)
#   - minor_error: Off by a small amount (0.5% - 5% relative)
#   - major_error: Significantly wrong (> 5% relative) — could mislead
#   - fabricated:  Number has no basis in source data at all
# ──────────────────────────────────────────────────────────────

def verify_numeric_claims(summary_id, source_data, numeric_claims):
    """
    Verify all numeric claims in a summary against source data.
    Returns a list of verification results with severity levels.
    """
    results = []

    for metric, values in numeric_claims.items():
        stated = values["stated"]
        actual = values["actual"]

        if isinstance(stated, (int, float)) and isinstance(actual, (int, float)):
            # Calculate how far off the stated value is from actual
            if actual != 0:
                relative_error = abs(stated - actual) / abs(actual)  # As a fraction
            else:
                relative_error = abs(stated - actual)

            absolute_error = abs(stated - actual)

            # Classify severity based on how far off the stated value is
            if relative_error <= 0.005:       # Within 0.5% — acceptable rounding
                severity = "exact"
            elif relative_error <= 0.05:      # Within 5% — minor but notable
                severity = "minor_error"
            else:                              # More than 5% off — major problem
                severity = "major_error"

            results.append({
                "metric": metric,
                "stated": stated,
                "actual": actual,
                "absolute_error": absolute_error,
                "relative_error_pct": relative_error * 100,
                "severity": severity
            })
        else:
            # Non-numeric comparison (e.g., country count: 15 vs 12)
            if stated != actual:
                results.append({
                    "metric": metric,
                    "stated": stated,
                    "actual": actual,
                    "absolute_error": abs(stated - actual) if isinstance(stated, (int, float)) else "N/A",
                    "relative_error_pct": "N/A",
                    "severity": "fabricated"  # The number was made up
                })
            else:
                results.append({
                    "metric": metric,
                    "stated": stated,
                    "actual": actual,
                    "absolute_error": 0,
                    "relative_error_pct": 0,
                    "severity": "exact"
                })

    return results


# ──────────────────────────────────────────────────────────────
# Run numeric verification on ALL summaries
# ──────────────────────────────────────────────────────────────
print("=" * 80)
print("NUMERIC REASONING CHECK RESULTS")
print("=" * 80)

numeric_results_all = []

for item in financial_dataset:
    results = verify_numeric_claims(
        item["id"], item["source_data"], item["numeric_claims"]
    )
    numeric_results_all.append({
        "id": item["id"],
        "company": item["company"],
        "checks": results
    })

    # Collect just the errors (not exact matches)
    errors = [r for r in results if r["severity"] in ("minor_error", "major_error", "fabricated")]

    print(f"\n{'─' * 60}")
    print(f"Summary {item['id']}: {item['company']}")
    print(f"  Total numeric claims checked: {len(results)}")
    print(f"  ✅ Exact/correct: {sum(1 for r in results if r['severity'] == 'exact')}")
    print(f"  ⚠️  Minor errors:  {sum(1 for r in results if r['severity'] == 'minor_error')}")
    print(f"  ❌ Major errors:  {sum(1 for r in results if r['severity'] == 'major_error')}")
    print(f"  🔴 Fabricated:    {sum(1 for r in results if r['severity'] == 'fabricated')}")

    # Show error details so you can see exactly what went wrong
    if errors:
        print(f"\n  ERRORS DETAIL:")
        for e in errors:
            if isinstance(e["relative_error_pct"], (int, float)):
                print(f"    [{e['severity'].upper()}] {e['metric']}: stated {e['stated']}, actual {e['actual']} (error: {e['relative_error_pct']:.1f}%)")
            else:
                print(f"    [{e['severity'].upper()}] {e['metric']}: stated {e['stated']}, actual {e['actual']}")

### Checkpoint ✅
You should see numeric verification results for all summaries. Key findings:
- **Summary 3 (MedVault)**: Major error on YoY growth — stated 25.0%, actual 20.0% (25% relative error)
- **Summary 5 (NexaPay)**: Major error on EPS growth — stated 22.0%, actual ~11.4% (93% relative error!). Also fabricated 15 countries instead of actual 12
- Most other summaries should have minimal or rounding-level errors

### Expected Output (partial)
```
Summary 3: MedVault Health
  ERRORS DETAIL:
    [MAJOR_ERROR] yoy_growth_pct: stated 25.0, actual 20.0 (error: 25.0%)

Summary 5: NexaPay Financial
  ERRORS DETAIL:
    [MAJOR_ERROR] eps_growth_pct: stated 22.0, actual 11.4 (error: 93.0%)
    [FABRICATED] countries_expanded: stated 15, actual 12
```

### 🔍 What Just Happened?
We built the **third layer**: numeric reasoning checks. This is specialized — it goes deeper than general factuality by computing exact relative errors and classifying them by severity. The 25% vs 20% error in MedVault looks small in absolute terms (5 percentage points) but represents a **25% relative error** — material enough to change an investment thesis. The NexaPay EPS growth error (22% vs 11.4%) is a **93% relative error** — nearly double the actual growth. These are exactly the kinds of errors that LLMs produce because they *predict plausible numbers* rather than calculate them.

Now we have three automated evaluation layers. The final layer takes a different approach: instead of checking specific claims, we'll ask an LLM to act as a **holistic judge**.

---

## Task 7: 🧠 Your Turn — Design a Structured Rubric for LLM-as-Judge ⚖️

### Goal
Design a **structured scoring rubric** that an LLM can use to evaluate the quality of financial summaries. This rubric will be used as part of a judge prompt.

### Why Rubric Design Matters

> 🎯 **Why This Matters:** The LLM-as-judge technique is powerful but only as good as its rubric. A vague rubric ("rate the quality from 1–5") produces vague, inconsistent scores. A precise rubric with specific, observable criteria produces reliable, actionable evaluations.

### How LLM-as-Judge Works

```
┌───────────────┐    ┌──────────────────┐    ┌───────────────┐
│  Source Data   │───▶│                  │    │  Structured   │
│  (Ground Truth)│    │    LLM Judge     │───▶│  Score +      │
│               │    │  (e.g., GPT-4)   │    │  Justification│
│  Generated    │───▶│                  │    └───────────────┘
│  Summary      │    │                  │
│               │    └──────────────────┘
│  Scoring      │           ▲
│  RUBRIC       │───────────┘
└───────────────┘
```

The quality of the rubric determines the quality of the judge. Think of it like a grading rubric for an exam — specific criteria lead to consistent grading.

### 🧠 Your Challenge

Design a rubric with **exactly 4 scoring dimensions**. For each dimension:

1. **Name** the dimension
2. **Define** what it measures (1–2 sentences)
3. **Provide scoring levels** (1–5) with specific, observable criteria for at least levels 1, 3, and 5
4. **Give an anchor example** — describe what a score of 1 and a score of 5 look like for a financial summary

**Requirements:**
- At least one dimension must cover factual accuracy
- At least one dimension must cover risk/negative information handling
- Dimensions should be independent (not overlapping)

**Hint:** Think about what matters to a portfolio manager reading these summaries. What would make them trust or distrust a summary?

> **Do NOT look ahead for the answer.** Design your rubric first. You'll use it (or a reference version) in the next task.

### ✍️ Your Rubric Design

**Write your rubric here** (double-click to edit this cell):

#### Dimension 1: ____________________
- **Measures:** ...
- **Score 1:** ...
- **Score 3:** ...
- **Score 5:** ...
- **Anchor (1):** ...
- **Anchor (5):** ...

#### Dimension 2: ____________________
- **Measures:** ...
- **Score 1:** ...
- **Score 3:** ...
- **Score 5:** ...
- **Anchor (1):** ...
- **Anchor (5):** ...

#### Dimension 3: ____________________
- **Measures:** ...
- **Score 1:** ...
- **Score 3:** ...
- **Score 5:** ...
- **Anchor (1):** ...
- **Anchor (5):** ...

#### Dimension 4: ____________________
- **Measures:** ...
- **Score 1:** ...
- **Score 3:** ...
- **Score 5:** ...
- **Anchor (1):** ...
- **Anchor (5):** ...

---
*You will use this rubric (or a reference version) in the next task.*

---

You've designed your rubric — now let's see it in action. This task implements the LLM-as-judge pattern, the **fourth and final layer** of our evaluation pipeline.

## Task 8: Build and Run the LLM Judge Evaluator ⚖️

### Goal
Implement an LLM-as-judge using a structured rubric. For reliability and reproducibility in this lab, we use **mock judge responses** that simulate what GPT-4 or Claude would return. This lets you focus on the evaluation pipeline design rather than API costs.

### Why Use an LLM Judge *in Addition* to Automated Checks?

> 💡 **Key Insight:** Automated checks (factuality, completeness, numeric) catch specific, measurable errors. But they can't assess **holistic quality** — things like "does this summary give the reader an accurate impression of the company's situation?" That's where an LLM judge excels. It can evaluate nuances like framing bias, tone, and overall coherence.

### Steps
1. Define the judge rubric (a reference implementation)
2. Build the judge evaluation function
3. Run the judge on all summaries
4. Compare judge scores to human scores

### Why Mock Responses?
In production, you'd call GPT-4 or Claude with the rubric prompt. Here we use mock responses because:
- Results are deterministic and reproducible
- No API key or cost needed
- You can focus on rubric design and interpretation
- The mock responses model realistic LLM judge behavior (including some disagreements with humans)

In [ ]:
# ──────────────────────────────────────────────────────────────
# REFERENCE RUBRIC FOR LLM-AS-JUDGE
# This is what you would include in your prompt to GPT-4/Claude.
# Compare this to the rubric YOU designed in Task 7!
#
# Key design principles:
#   1. Each dimension is INDEPENDENT (no overlap)
#   2. Scoring levels use OBSERVABLE criteria (not vague adjectives)
#   3. Both low and high scores are clearly defined
#   4. The rubric covers different failure modes we've seen
# ──────────────────────────────────────────────────────────────

JUDGE_RUBRIC = """
You are an expert financial analyst evaluating AI-generated quarterly earnings summaries.

Score the summary on these 4 dimensions (1-5 each):

## Dimension 1: Factual Accuracy
- 1: Multiple incorrect facts or numbers; core figures wrong
- 2: One or more significant factual errors
- 3: Facts are mostly correct; minor rounding or imprecision
- 4: All stated facts are correct; negligible imprecision
- 5: All stated facts are verifiably correct and precisely stated

## Dimension 2: Completeness & Risk Coverage
- 1: Misses most key information; significant omissions of risk factors
- 2: Covers basics but omits important negatives or risk signals
- 3: Covers main financial metrics; may miss some context or risk factors
- 4: Good coverage including most risk factors and notable events
- 5: Comprehensive coverage; all material facts, risks, and events included

## Dimension 3: Numeric Reasoning
- 1: Growth rates or calculations are significantly wrong (>5% off)
- 2: One or more calculation errors or unsupported numeric claims
- 3: Most calculations correct; minor rounding acceptable
- 4: All calculations verifiable and correct within rounding
- 5: Calculations are precise; growth rates, margins, changes all verified

## Dimension 4: Balanced Framing
- 1: Misleadingly positive or negative; framing contradicts the data story
- 2: Notable bias in framing; downplays negatives or overhypes positives
- 3: Generally balanced; some framing choices could be improved
- 4: Fair representation of both positive and negative trends
- 5: Exemplary balance; reader gets accurate overall impression

For each dimension, provide:
- Score (1-5)
- Brief justification (1-2 sentences)
- Overall quality score (average of 4 dimensions)
"""

print("📋 Judge Rubric Defined")
print(f"   Dimensions: 4 (Factual Accuracy, Completeness, Numeric Reasoning, Balanced Framing)")
print(f"   Scale: 1-5 per dimension")
print(f"   Total possible score: 20")
print(f"\n   Notice how each dimension maps to a different evaluation layer:")
print(f"   • Factual Accuracy  → Task 4 (Claim verification)")
print(f"   • Completeness      → Task 5 (Missing fact detection)")
print(f"   • Numeric Reasoning → Task 6 (Calculation checks)")
print(f"   • Balanced Framing  → NEW dimension the judge adds!")

In [ ]:
# ──────────────────────────────────────────────────────────────
# MOCK LLM JUDGE RESPONSES
# In production, these would come from calling GPT-4/Claude with:
#   prompt = JUDGE_RUBRIC + source_data + generated_summary
# These mocks are calibrated to be REALISTIC — including deliberate
# disagreements with human scores, which you'll investigate in Task 9.
# ──────────────────────────────────────────────────────────────

MOCK_JUDGE_SCORES = {
    1: {  # TechFlow Inc. — high quality summary ✅
        "factual_accuracy": {"score": 5, "justification": "All stated revenue, EPS, growth rates, and event details match source data precisely."},
        "completeness_risk": {"score": 5, "justification": "Comprehensive coverage of financial metrics, both acquisitions, guidance, and balance sheet. No material omissions."},
        "numeric_reasoning": {"score": 4, "justification": "Growth rates are correctly calculated within rounding. QoQ growth of 10.9% matches 10.94% actual."},
        "balanced_framing": {"score": 5, "justification": "Summary accurately conveys the positive quarter without hype. Factual tone throughout."},
        "overall": 4.75
    },
    2: {  # GreenLeaf Energy — omits critical negatives ⚠️
        "factual_accuracy": {"score": 4, "justification": "Stated facts are correct — revenue, margins, EPS all match. No false statements."},
        "completeness_risk": {"score": 1, "justification": "Critical omissions: QoQ revenue decline, net income collapse, debt covenant waiver, CEO transition, debt level. Fundamentally incomplete."},
        "numeric_reasoning": {"score": 4, "justification": "YoY growth calculation of 8.5% is correct (8.54% actual). Other numbers accurate."},
        "balanced_framing": {"score": 1, "justification": "Misleadingly positive framing. Phrases like 'healthy margin' and 'confidence in growth strategy' while company is deteriorating. Omits all negative signals."},
        "overall": 2.50
    },
    3: {  # MedVault Health — YoY number is wrong ❌
        "factual_accuracy": {"score": 3, "justification": "Most facts correct but states 25% YoY growth when actual is ~20%. This is a material misstatement."},
        "completeness_risk": {"score": 5, "justification": "Excellent coverage: revenue, growth, margins, EPS, events, balance sheet, guidance, and operating expenses all addressed."},
        "numeric_reasoning": {"score": 2, "justification": "YoY growth stated as 25% but actual is 20.0% — a significant calculation error that materially misrepresents growth."},
        "balanced_framing": {"score": 4, "justification": "Generally balanced. Notes operating expense growth. Tone appropriate for a strong quarter."},
        "overall": 3.50
    },
    4: {  # UrbanLogix — omits bad news, positive spin ⚠️
        "factual_accuracy": {"score": 3, "justification": "Revenue and EPS correct. But framing operating losses as 'strategic investments' without stating the net loss amount is borderline misleading."},
        "completeness_risk": {"score": 2, "justification": "Omits 15% workforce restructuring, worsening losses (-$3.2M to -$8.5M), and high debt-to-cash ratio. Only mentions CityGrid partnership."},
        "numeric_reasoning": {"score": 4, "justification": "Stated numbers are accurate: revenue $67M, YoY 6.3%, expenses $72M, EPS -$0.62."},
        "balanced_framing": {"score": 1, "justification": "Heavily positive spin. Uses 'resilience', 'strategic investments', 'exciting partnership' while company has widening losses and is laying off 15% of staff."},
        "overall": 2.50
    },
    5: {  # NexaPay — EPS growth wrong, country count wrong ❌
        "factual_accuracy": {"score": 3, "justification": "Most figures correct, but states 15 new countries (actual: 12) and EPS growth of 22% (actual: ~11.4%). Two distinct factual errors."},
        "completeness_risk": {"score": 4, "justification": "Good coverage of most metrics, events, and guidance. Notes risk about opex growth. Minor: doesn't mention net income growth rate."},
        "numeric_reasoning": {"score": 2, "justification": "EPS growth calculation is nearly double the actual (22% vs 11.4%). Country count is fabricated. Two significant numeric errors."},
        "balanced_framing": {"score": 4, "justification": "Includes a risk note about opex growing faster than revenue. Otherwise positive but fair given strong fundamentals."},
        "overall": 3.25
    },
    6: {  # SkyBridge Telecom — solid but missing debt info
        "factual_accuracy": {"score": 5, "justification": "All stated facts and numbers are accurate. Revenue, growth rates, EPS, margins, events all match source data."},
        "completeness_risk": {"score": 3, "justification": "Covers main metrics and events well but omits the $340M debt position, which is significant for a telecom. Cash mentioned but not in context of debt."},
        "numeric_reasoning": {"score": 5, "justification": "All calculations correct: QoQ 1.6% (actual 1.56%), YoY 3.7% (actual 3.72%). Well within rounding."},
        "balanced_framing": {"score": 4, "justification": "Appropriately measured tone for a steady-growth quarter. 'Modest' and 'steady' are fair descriptors."},
        "overall": 4.25
    }
}


def run_llm_judge(summary_id):
    """
    Simulate running an LLM judge on a financial summary.
    In production, this would call GPT-4/Claude with the rubric prompt.
    Returns the multi-dimensional score + justifications.
    """
    return MOCK_JUDGE_SCORES.get(summary_id, None)


# ──────────────────────────────────────────────────────────────
# Run the judge on all summaries and compare to human scores
# ──────────────────────────────────────────────────────────────
print("=" * 80)
print("LLM JUDGE EVALUATION RESULTS")
print("=" * 80)

judge_results = []

for item in financial_dataset:
    scores = run_llm_judge(item["id"])
    if scores:
        judge_results.append({
            "id": item["id"],
            "company": item["company"],
            "human_score": item["human_quality_score"],
            **scores
        })

        print(f"\n{'─' * 60}")
        print(f"Summary {item['id']}: {item['company']}")
        print(f"  Factual Accuracy:      {scores['factual_accuracy']['score']}/5 — {scores['factual_accuracy']['justification']}")
        print(f"  Completeness & Risk:   {scores['completeness_risk']['score']}/5 — {scores['completeness_risk']['justification']}")
        print(f"  Numeric Reasoning:     {scores['numeric_reasoning']['score']}/5 — {scores['numeric_reasoning']['justification']}")
        print(f"  Balanced Framing:      {scores['balanced_framing']['score']}/5 — {scores['balanced_framing']['justification']}")
        print(f"  📊 Judge Overall:      {scores['overall']}/5")
        print(f"  👤 Human Score:        {item['human_quality_score']}/5")
        # Check if judge and human agree (within 0.75 points)
        agreement = "✅ AGREE" if abs(scores['overall'] - item["human_quality_score"]) <= 0.75 else "⚠️  DISAGREE"
        print(f"  {agreement} (gap: {abs(scores['overall'] - item['human_quality_score']):.2f})")

### Checkpoint ✅
You should see judge scores for all 6 summaries across 4 dimensions, plus comparison to human scores. Key observations:
- **TechFlow (Summary 1)**: Judge and human should agree (both high)
- **GreenLeaf (Summary 2)**: Judge and human should agree (both low) — but for the right reasons? Check the dimension breakdowns
- **All six** summaries land within 0.5 points of the human score. But look at the *sign* of the gaps: the judge is **higher than the human on 5 of 6** summaries. That pattern sets up the Open Challenge in Task 9

### Expected Output (partial)
```
Summary 2: GreenLeaf Energy
  Factual Accuracy:      4/5 — Stated facts are correct...
  Completeness & Risk:   1/5 — Critical omissions...
  Balanced Framing:      1/5 — Misleadingly positive...
  📊 Judge Overall:      2.50/5
  👤 Human Score:        2/5
  ✅ AGREE (gap: 0.50)
```

### 🔍 What Just Happened?
We ran an LLM judge — the **fourth and final layer** of our pipeline. Notice something powerful: the judge produced **dimensional scores with justifications**, not just a single number. For GreenLeaf, the judge gave 4/5 on factual accuracy but 1/5 on completeness and framing. This granularity tells us *exactly what's wrong*, not just "it's bad." But the judge isn't perfect — in Task 9, you'll investigate where it disagrees with human experts and why.

---

With a ±0.75 tolerance, the judge "agrees" with humans on **all 6** summaries (Pearson r ≈ 0.99). Case closed? Not quite. Agreement on an overall number can hide two real problems: a **systematic bias** (the gaps are almost all positive) and **disagreement about reasons** (dimension scores that don't match the human's concerns). No automated evaluation is perfect; your job is to find *how* this one is imperfect.

## Task 9: ⚠️ Open Challenge — Where Does the LLM Judge Disagree with Humans?

### Goal
Investigate the cases where the LLM judge's overall score **disagrees** with the human expert score. Understand *why* they disagree and what this reveals about the strengths and limitations of LLM-as-judge evaluation.

> 🎯 **Why This Matters:** If you deploy an LLM judge in production, you need to know where it's reliable and where it's not. Blind trust in automated evaluation is just as dangerous as blind trust in generated content.

### 🧪 Experiment + ⚠️ Open Challenge

This task combines experimentation with open-ended analysis.

**Step 1:** Run the comparison analysis below.

**Step 2:** Look at the **direction** of every gap, then re-run the cell with a stricter tolerance (change `0.75` to `0.25`) to surface the closest calls. Also compare SkyBridge's judge score (≈ 4.25) and human score (4) with its **pipeline decision in Task 10**.

**Step 3:** For each disagreement (or near-disagreement), answer these questions in the markdown cell:

1. **Who is more "right" — the judge or the human?** Justify your answer using the source data.
2. **What does this disagreement reveal** about LLM-as-judge limitations?
3. **How would you fix the rubric** to reduce this type of disagreement?
4. **Can you find a specific claim or detail** in the summary where the judge's dimensional scores seem inconsistent or where a different rubric would change the outcome?

> **This is an open challenge.** There may be multiple valid interpretations. The goal is to reason carefully about automated evaluation reliability.

In [ ]:
# ──────────────────────────────────────────────────────────────
# JUDGE vs HUMAN COMPARISON
# This is the "meta-evaluation" — evaluating our evaluator!
# We compare the LLM judge's scores to human expert scores
# to understand where automated evaluation is reliable vs not.
# ──────────────────────────────────────────────────────────────
print("=" * 80)
print("JUDGE vs HUMAN COMPARISON")
print("=" * 80)
print(f"\n{'Company':25s} | {'Human':>5s} | {'Judge':>5s} | {'Gap':>5s} | {'Agreement':>12s}")
print("─" * 70)

disagreements = []
for jr in judge_results:
    gap = jr["overall"] - jr["human_score"]
    # Agreement threshold: within 0.75 points = agree; beyond = disagree
    agreement = "✅ Agree" if abs(gap) <= 0.75 else "⚠️  Disagree"
    print(f"  {jr['company']:23s} | {jr['human_score']:>5.1f} | {jr['overall']:>5.2f} | {gap:>+5.2f} | {agreement}")
    if abs(gap) > 0.75:
        disagreements.append(jr)

# Compute Pearson correlation — how linearly related are judge and human scores?
human_scores = [jr["human_score"] for jr in judge_results]
judge_scores = [jr["overall"] for jr in judge_results]
correlation = np.corrcoef(human_scores, judge_scores)[0, 1]

print(f"\n📊 Overall Correlation (Pearson): {correlation:.3f}")
print(f"   Agreements: {len(judge_results) - len(disagreements)}/{len(judge_results)}")
print(f"   Disagreements: {len(disagreements)}/{len(judge_results)}")

# Deep-dive into disagreements — these are the interesting cases
if disagreements:
    print(f"\n{'=' * 80}")
    print("DISAGREEMENT DEEP DIVE")
    print(f"{'=' * 80}")
    for d in disagreements:
        print(f"\n🔍 {d['company']} — Human: {d['human_score']}/5, Judge: {d['overall']}/5")
        print(f"   Dimension breakdown:")
        print(f"     Factual Accuracy:    {d['factual_accuracy']['score']}/5")
        print(f"     Completeness & Risk: {d['completeness_risk']['score']}/5")
        print(f"     Numeric Reasoning:   {d['numeric_reasoning']['score']}/5")
        print(f"     Balanced Framing:    {d['balanced_framing']['score']}/5")
        print(f"   Judge justifications:")
        for dim in ["factual_accuracy", "completeness_risk", "numeric_reasoning", "balanced_framing"]:
            print(f"     {dim}: {d[dim]['justification']}")

### ✍️ Your Disagreement Analysis

**Write your analysis here** (double-click to edit this cell):

1. **Which summary has the biggest disagreement?**

   *Your answer...*

2. **Who is more "right" — the judge or the human? Why?**

   *Your answer...*

3. **What limitation of LLM-as-judge does this reveal?**

   *Your answer...*

4. **How would you modify the rubric to fix this?**

   *Your answer...*

---

---

You've now built and analyzed all four evaluation layers independently. The final step is what makes this a real **production pipeline**: combining everything into a single, actionable report.

## Task 10: Combine All Results and Recommend Improvements 📋

### Goal
Bring together factuality, completeness, numeric reasoning, and judge scores into a **unified evaluation report**. Use this to make concrete recommendations about which summaries are safe vs unsafe to publish, and what improvements to prioritize.

> 🎯 **Why This Matters:** Individual evaluation dimensions are useful for diagnosis, but decision-makers need a clear **PASS / REVIEW / FAIL** signal. This combined report turns evaluation data into action.

### Steps
1. Run the combined analysis
2. Interpret the results
3. Make recommendations

In [ ]:
# ──────────────────────────────────────────────────────────────
# COMBINED EVALUATION REPORT
# This brings together ALL four evaluation layers into a single
# PASS / REVIEW / FAIL decision for each summary.
#
# Decision logic:
#   🔴 FAIL if:   major numeric error, critical omission, OR judge < 2.5
#   🟡 REVIEW if: judge < 3.5 (borderline quality)
#   🟢 PASS if:   no major issues AND judge >= 3.5
# ──────────────────────────────────────────────────────────────
print("=" * 80)
print("COMBINED EVALUATION REPORT")
print("=" * 80)

combined_report = []

for item in financial_dataset:
    sid = item["id"]

    # Gather results from each evaluation layer
    fact = next(f for f in factuality_results if f["id"] == sid)       # Layer 1: Factuality
    comp = next(c for c in completeness_results if c["id"] == sid)     # Layer 2: Completeness
    nums = next(n for n in numeric_results_all if n["id"] == sid)      # Layer 3: Numeric
    judge = next(j for j in judge_results if j["id"] == sid)           # Layer 4: Judge

    # Count major numeric errors (the most dangerous kind)
    numeric_errors = sum(1 for c in nums["checks"] if c["severity"] in ("major_error", "fabricated"))

    # ── Apply decision logic ──
    # Any single critical failure = FAIL (defense in depth)
    has_major_numeric = numeric_errors > 0
    has_critical_omission = len(comp["critical_missing"]) > 0
    judge_low = judge["overall"] < 2.5
    judge_medium = judge["overall"] < 3.5

    if has_major_numeric or has_critical_omission or judge_low:
        decision = "🔴 FAIL"
        reason_parts = []
        if has_major_numeric:
            reason_parts.append(f"{numeric_errors} major numeric error(s)")
        if has_critical_omission:
            reason_parts.append(f"{len(comp['critical_missing'])} critical omission(s)")
        if judge_low:
            reason_parts.append(f"judge score {judge['overall']:.2f} < 2.5")
        reason = "; ".join(reason_parts)
    elif judge_medium:
        decision = "🟡 REVIEW"
        reason = f"Judge score {judge['overall']:.2f} suggests quality concerns"
    else:
        decision = "🟢 PASS"
        reason = "All checks passed"

    report_entry = {
        "id": sid,
        "company": item["company"],
        "factuality_score": fact["factuality_score"],
        "completeness_score": comp["overall_score"],
        "numeric_errors": numeric_errors,
        "critical_omissions": len(comp["critical_missing"]),
        "judge_overall": judge["overall"],
        "human_score": item["human_quality_score"],
        "decision": decision,
        "reason": reason
    }
    combined_report.append(report_entry)

# ── Print the combined report table ──
print(f"\n{'Company':25s} | {'Fact':>5s} | {'Comp':>5s} | {'#Err':>4s} | {'Judge':>5s} | {'Human':>5s} | {'Decision':12s}")
print("─" * 95)
for r in combined_report:
    print(f"  {r['company']:23s} | {r['factuality_score']:>5.3f} | {r['completeness_score']:>5.3f} | {r['numeric_errors']:>4d} | {r['judge_overall']:>5.2f} | {r['human_score']:>5d} | {r['decision']}")
    if "FAIL" in r["decision"] or "REVIEW" in r["decision"]:
        print(f"  {'':23s}   Reason: {r['reason']}")

# ── Summary statistics ──
pass_count = sum(1 for r in combined_report if "PASS" in r["decision"])
review_count = sum(1 for r in combined_report if "REVIEW" in r["decision"])
fail_count = sum(1 for r in combined_report if "FAIL" in r["decision"])

print(f"\n{'=' * 80}")
print(f"SUMMARY: {pass_count} PASS | {review_count} REVIEW | {fail_count} FAIL out of {len(combined_report)} summaries")
print(f"{'=' * 80}")

In [ ]:
# ──────────────────────────────────────────────────────────────
# FAILURE PATTERN ANALYSIS
# Look across ALL summaries to find the most common issues.
# This tells us what to fix in the LLM generation system itself,
# not just which individual summaries to block.
# ──────────────────────────────────────────────────────────────
print("=" * 80)
print("FAILURE PATTERN ANALYSIS")
print("=" * 80)

print("\n📋 Most common issues across all summaries:\n")

# Count how many summaries have each type of issue
issue_counts = defaultdict(int)
for r in combined_report:
    if r["numeric_errors"] > 0:
        issue_counts["Numeric calculation errors"] += 1
    if r["critical_omissions"] > 0:
        issue_counts["Critical information omitted"] += 1
    if r["factuality_score"] < 0.3:
        issue_counts["Low factuality score"] += 1
    if r["completeness_score"] < 0.5:
        issue_counts["Low completeness score"] += 1
    if r["judge_overall"] < 3.0:
        issue_counts["Judge score below 3.0"] += 1

# Show issues sorted by frequency (most common first)
for issue, count in sorted(issue_counts.items(), key=lambda x: -x[1]):
    print(f"  {count}/{len(combined_report)} summaries — {issue}")

# Actionable recommendations based on the patterns we found
print("\n📋 Recommended improvements (prioritized):\n")
print("  1. ADD NUMERIC VERIFICATION GATE: Auto-check all growth rates,")
print("     margins, and calculations before publishing. Block summaries")
print("     with any major numeric error.")
print("")
print("  2. ADD COMPLETENESS CHECKLIST: Require coverage of all material")
print("     facts, especially: risk events, negative trends, and balance")
print("     sheet concerns. Flag omissions of critical items.")
print("")
print("  3. ADD FRAMING BIAS DETECTION: Check for misleadingly positive")
print("     framing when underlying data shows negative trends.")
print("")
print("  4. CALIBRATE LLM JUDGE: Compare judge scores to human scores")
print("     across more samples. Adjust rubric for areas of disagreement.")
print("")
print("  5. IMPLEMENT TIERED REVIEW: Auto-pass clean summaries,")
print("     flag borderline cases for quick human review,")
print("     block clearly problematic summaries.")

### Checkpoint ✅
You should see a combined evaluation report with pass/review/fail decisions for each summary. Expected pattern:
- **TechFlow**: 🟢 PASS (the only one)
- **SkyBridge**: 🔴 FAIL on **1 critical omission** (its $340M debt position), even though both the judge (≈ 4.25) and the human (4/5) rated it well. A hard rule on critical omissions is **stricter than either holistic score**. Is that what you want? Discuss in Task 9
- **MedVault** and **NexaPay**: 🔴 FAIL (major numeric errors)
- **GreenLeaf** and **UrbanLogix**: 🔴 FAIL (critical omissions)

The failure pattern analysis should show that numeric errors and critical omissions are the two most common issues.

### 🔍 What Just Happened?
You just built a **complete, multi-layered evaluation pipeline** for generative outputs! Each layer catches different kinds of errors:

| Layer | What It Catches | Example |
|-------|----------------|---------|
| Factuality | Wrong claims | MedVault's 25% growth claim |
| Completeness | Missing information | GreenLeaf's hidden risks |
| Numeric | Math errors | NexaPay's doubled EPS growth |
| LLM Judge | Holistic quality | UrbanLogix's misleading framing |

No single layer would have caught everything. That's why **defense in depth** matters for evaluation — the same principle that makes security systems robust.

---

## ✅ Validation and Success Criteria

You have successfully completed this lab if:

- [ ] All 6 financial summaries are loaded and explorable
- [ ] Factuality scoring identifies contradicted and fabricated claims
- [ ] You designed completeness criteria (Task 5) with your own reasoning
- [ ] Completeness scoring flags critical omissions in GreenLeaf and UrbanLogix
- [ ] Numeric reasoning checks catch the major errors in MedVault and NexaPay
- [ ] You designed a rubric (Task 7) with 4 scoring dimensions
- [ ] LLM judge evaluator runs and produces scored results
- [ ] You analyzed at least one judge-vs-human disagreement (Task 9)
- [ ] Combined report produces pass/review/fail decisions with reasons
- [ ] You can explain why GreenLeaf gets a low score despite having correct facts

---

## 🔧 Troubleshooting

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| `ModuleNotFoundError` | Package not installed | Re-run the `pip install` cell, restart kernel |
| Factuality scores all 0 | Claim extraction returning empty | Check that `extract_claims()` is splitting on sentence boundaries correctly |
| Completeness scores all 1.0 | Keyword matching too loose | Check `score_completeness()` overlap threshold |
| Judge scores missing | `MOCK_JUDGE_SCORES` dictionary key mismatch | Verify summary IDs match between dataset and mock scores |
| Combined report shows all PASS | Decision thresholds too lenient | Check the threshold logic in the combined report cell |

---

## 🧩 Stretch Goal: Debug a Flawed Evaluation

If you have extra time, try this debug challenge:

The factuality scorer uses **keyword matching** to verify claims. This approach has known weaknesses:

1. **Find a false positive**: Identify a claim that the scorer marks as "supported" but that a human would question.
2. **Find a false negative**: Identify a claim that the scorer marks as "unsupported" but that is clearly verifiable from the source data.
3. **Propose a fix**: How would you improve the claim verification to handle these cases? Consider: What would an LLM-based verifier do differently?

---

## 🎯 Key Takeaways

1. 🔍 **Factuality ≠ completeness**: A summary can be 100% factually correct and still be dangerously misleading by omitting critical information (GreenLeaf Energy).

2. 🔢 **Numeric reasoning needs explicit verification**: LLMs generate plausible-looking numbers that may be wrong. Always verify calculations against source data (MedVault's 25% vs actual 20% YoY growth).

3. 📋 **Rubric quality drives judge quality**: A vague rubric produces inconsistent LLM judge scores. Specific, observable criteria with anchor examples make evaluation reliable and actionable.

4. 🧩 **Multi-dimensional evaluation is essential**: No single metric captures overall summary quality. Combining factuality, completeness, numeric accuracy, and framing bias gives a complete picture.

5. ⚖️ **LLM judges have limitations**: They can disagree with humans, especially on nuanced issues like framing bias and materiality judgments. Always calibrate against human evaluation data.

6. 🚀 **Evaluation informs action**: The goal isn't just scores — it's decisions. A good evaluation pipeline tells you which summaries to publish, which to review, and what to fix in the generation system.

---

## 🔮 What's Next?

In **Module 06: Evaluating Agentic Systems**, you'll take evaluation to the next level — moving from evaluating *static text outputs* to evaluating *multi-step agent behavior*. You'll learn how to:

- Evaluate agents that take **actions** (tool calls, API requests), not just generate text
- Assess **multi-turn reasoning** chains for correctness and safety
- Build evaluation frameworks for agent **trajectories** (sequences of decisions)
- Handle the unique challenges of evaluating systems that interact with the real world

If this module taught you to check "did the LLM say the right thing?", Module 06 will teach you to check "did the agent *do* the right thing?" 🚀

---

## 🏋️ Exercises

Practice generative-output evaluation: factuality, completeness, and rubric/LLM-as-judge scoring.

### Exercise 1 — Beginner: Keyword completeness
Score completeness as the fraction of expected key points present in an answer.

<details>
<summary>Show solution</summary>

```python
def completeness(answer, key_points):
    a = answer.lower()
    return sum(kp.lower() in a for kp in key_points) / len(key_points)
print(completeness("50 MB free and 500 MB enterprise", ["50 MB", "500 MB"]))
```
</details>

### Exercise 2 — Applied: Build a rubric scorer
Average several 1–5 rubric dimensions into a normalized 0–1 score.

<details>
<summary>Show solution</summary>

```python
def rubric_score(scores):  # scores: dict dim->1..5
    return sum(scores.values()) / (5 * len(scores))
print(rubric_score({"factuality": 5, "completeness": 4, "clarity": 5}))
```
</details>

### Exercise 3 — Challenge: Mock LLM-as-judge
Write a judge function returning {"verdict","reason"} based on whether the answer contains both numbers.

<details>
<summary>Show solution</summary>

```python
def judge(answer):
    ok = "50 mb" in answer.lower() and "500 mb" in answer.lower()
    return {"verdict": "pass" if ok else "fail",
            "reason": "Both tiers stated" if ok else "Missing a tier"}
print(judge("50 MB free, 500 MB enterprise"))
```
</details>

### Exercise 4 — Stretch: Compare two judges
Score 5 answers with a strict and a lenient judge and analyze where they disagree. (Open-ended.)
